# 3. Load prepared evidence and run semantic hybrid search
Start each run by resetting the configured demo index, then upload all records saved by notebook 1 and run semantic hybrid search. All previous documents in this index are removed, including other claims. Original files and the prepared input JSON are not deleted.

## Databricks Runtime setup
Run the package cell once, then continue after Python restarts. Set the `CLAIM_FOLDER` widget to the same input claim folder used in notebook 1; `DI_OUTPUT_ROOT` is the output location, not the input. Edit non-secret configuration below; credentials come from the approved secret scope. Use the same DI paths, embedding deployment, and index across all three notebooks.

In [ ]:
%pip install -r ../../../../../requirements.txt
dbutils.library.restartPython()

In [ ]:
import os
os.environ.setdefault('DATABRICKS_SECRET_SCOPE', 'rag-workshop')
os.environ.setdefault('AZURE_OPENAI_ENDPOINT', 'https://<openai-resource>.openai.azure.com/')
os.environ.setdefault('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large')
os.environ.setdefault('AZURE_OPENAI_EMBEDDING_DIMENSIONS', '3072')
os.environ.setdefault('AZURE_SEARCH_SERVICE_ENDPOINT', 'https://<search-service>.search.windows.net')
os.environ.setdefault('AZURE_SEARCH_DI_INDEX_NAME', 'claims-di-databricks-v1')
os.environ.setdefault('DI_OUTPUT_ROOT', '/Volumes/<catalog>/<schema>/<volume>/rag-output/di')
dbutils.widgets.text('CLAIM_FOLDER', os.getenv('CLAIM_FOLDER', '/Volumes/<catalog>/<schema>/<volume>/claims/claim-001'))
os.environ['CLAIM_FOLDER'] = dbutils.widgets.get('CLAIM_FOLDER')

In [ ]:
import os, json
from pathlib import Path, PurePosixPath
from urllib.parse import urlsplit, unquote
from tempfile import TemporaryDirectory
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.models import VectorizedQuery
from azure.storage.filedatalake import FileSystemClient
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
scope = os.environ['DATABRICKS_SECRET_SCOPE']
os.environ['AZURE_TENANT_ID'] = dbutils.secrets.get(scope=scope, key='azure-tenant-id')
os.environ['AZURE_CLIENT_ID'] = dbutils.secrets.get(scope=scope, key='azure-client-id')
os.environ['AZURE_CLIENT_SECRET'] = dbutils.secrets.get(scope=scope, key='azure-client-secret')
credential = DefaultAzureCredential()

claim_location = os.getenv('CLAIM_FOLDER', 'data/claims/claim-001')
claim_id = PurePosixPath(claim_location.replace('\\', '/').rstrip('/')).name
claim_filter = "claim_id eq '" + claim_id.replace("'", "''") + "'"
output_location = os.getenv('DI_OUTPUT_ROOT', 'output/claims/di').rstrip('/\\') + '/' + claim_id
index_name = os.getenv('AZURE_SEARCH_DI_INDEX_NAME', 'claims-di-v1')
azure_openai_model_dimensions = int(os.getenv('AZURE_OPENAI_EMBEDDING_DIMENSIONS', '3072'))
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_columns', 15)
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_seq_items', 3)

def ShowPreview(frame):
    preview = frame.head(5).copy()
    for column in preview.columns:
        if column.endswith('_vector'):
            preview[column] = preview[column].map(lambda vector: f'{len(vector)} dimensions')
        else:
            preview[column] = preview[column].map(lambda value: str(value)[:160])
    display(preview)

print('DI index:', index_name, '| Claim:', claim_id)

with SearchIndexClient(endpoint=os.environ['AZURE_SEARCH_SERVICE_ENDPOINT'], credential=credential) as index_client:
    index = index_client.get_index(index_name)
    index_client.delete_index(index_name)
    index_client.create_index(index)
print('Reset index:', index_name)

## Step 1: Read prepared data from local storage, a Volume, or ADLS
Read the JSON saved by notebook 1. Direct ADLS stages only that JSON; it does not download the original files or extraction cache.

In [ ]:
staging_folders, storage_clients = [], []
output_remote = None
if output_location.startswith('abfss://'):
    parsed = urlsplit(output_location)
    container, host = parsed.netloc.split('@')
    prefix = unquote(parsed.path).strip('/')
    output_client = FileSystemClient(account_url=f'https://{host}', file_system_name=container, credential=credential)
    storage_clients.append(output_client)
    staging = TemporaryDirectory(prefix='azure-rag-workshop-')
    staging_folders.append(staging)
    output_folder = Path(staging.name)
    with (output_folder / 'processed-evidence-v1.json').open('wb') as stream:
        output_client.get_file_client(prefix + '/processed-evidence-v1.json').download_file().readinto(stream)
    output_remote = (output_client, prefix)
else:
    output_folder = (ROOT / output_location).expanduser().resolve()
persisted_directories = set()

def PersistFile(path):
    if output_remote:
        client, prefix = output_remote
        remote_name = prefix + '/' + path.relative_to(output_folder).as_posix()
        parts = PurePosixPath(remote_name).parent.parts
        for count in range(1, len(parts) + 1):
            directory = '/'.join(parts[:count])
            if directory not in persisted_directories:
                directory_client = client.get_directory_client(directory)
                if not directory_client.exists():
                    directory_client.create_directory()
                persisted_directories.add(directory)
        with path.open('rb') as stream:
            client.get_file_client(remote_name).upload_data(stream, overwrite=True)

processed_file = output_folder / 'processed-evidence-v1.json'
print('Prepared file:', processed_file)

In [ ]:
records = json.loads(processed_file.read_text(encoding='utf-8'))
index_columns = ['id', 'claim_id', 'document_hash', 'chunk_id', 'title', 'header', 'content', 'source_type', 'content_kind', 'filename', 'source_path', 'source_files', 'parent_emails', 'figure_id', 'source_region', 'analyzer_id', 'analyzer_api_version', 'content_vector']
df = pd.DataFrame(records)
ShowPreview(df[index_columns])
print('Rows to upload:', len(df), '| Columns:', len(df.columns))

## Step 2: Initial load (Azure writes)
Upload every prepared record into the empty demo index. Save the SDK results and let upload failures surface; there is no incremental synchronization.

In [ ]:
search_endpoint = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
search_client = SearchClient(endpoint=search_endpoint, index_name=index_name, credential=credential)
upload_results = []
for start in range(0, len(records), 100):
    source_batch = records[start:start+100]
    batch = search_client.upload_documents(documents=source_batch)
    batch_results = [{'id': result.key, 'succeeded': result.succeeded, 'status_code': result.status_code, 'error': result.error_message} for result in batch]
    upload_results.extend(batch_results)
    report = output_folder / 'upload-results.json'
    report.write_text(json.dumps(upload_results, indent=2), encoding='utf-8')
    PersistFile(report)
    print('Batch:', start // 100 + 1, '| Accepted:', sum(result.succeeded for result in batch), '/', len(source_batch))
    if len(batch) != len(source_batch) or any(not result.succeeded for result in batch):
        ShowPreview(pd.DataFrame([row for row in batch_results if not row['succeeded']]))
        raise RuntimeError(batch_results)
print('Uploaded:', len(upload_results))

## Step 3: Preview the loaded records
Show a small sample and the loaded row count. Search visibility can lag briefly; rerun this preview cell if needed.

In [ ]:
visible_rows = list(search_client.search(search_text='*', select=['id', 'claim_id', 'filename', 'content_kind', 'chunk_id', 'figure_id']))
ShowPreview(pd.DataFrame(visible_rows))
print('Prepared rows:', len(records), '| Indexed rows:', len(visible_rows))

## Step 4: Ask a semantic hybrid retrieval question (paid query embedding)
Edit the question cell, then run the next cell. It makes one query-embedding call plus one Search request: text + 50 vector candidates, claim prefilter, and semantic reranking, returning at most five results. A semantic failure remains an error. Scores are ranking signals; captions are source excerpts, not a generated answer or a claims determination.

The query creates its own clients and remains rerunnable after cleanup. Keep the same embedding resource, deployment, and dimensions used for document vectors.

In [ ]:
query = 'What do the inspection photos and survey diagram show, and which sources support those observations?'

In [ ]:
from openai import OpenAI
embedding_deployment = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large')
foundry_openai_endpoint = os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')
search_endpoint = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
with DefaultAzureCredential() as query_credential:
    with OpenAI(base_url=f'{foundry_openai_endpoint}/openai/v1/', api_key=get_bearer_token_provider(query_credential, 'https://ai.azure.com/.default')) as query_client:
        embedding_response = query_client.embeddings.create(input=query, model=embedding_deployment, dimensions=azure_openai_model_dimensions)
    query_vector = embedding_response.data[0].embedding
    vector_query = VectorizedQuery(vector=query_vector, k_nearest_neighbors=50, fields='content_vector')
    with SearchClient(endpoint=search_endpoint, index_name=index_name, credential=query_credential) as query_search_client:
        hybrid_results = list(query_search_client.search(
            search_text=query, search_fields=['title', 'header', 'content', 'filename'], vector_queries=[vector_query],
            filter=claim_filter, vector_filter_mode='preFilter', query_type='semantic',
            semantic_configuration_name='claims-semantic', semantic_error_mode='fail',
            query_caption='extractive', query_caption_highlight_enabled=False,
            select=['id', 'claim_id', 'document_hash', 'chunk_id', 'title', 'header', 'content', 'filename', 'content_kind', 'source_path', 'source_files', 'parent_emails', 'figure_id', 'source_region'], top=5,
        ))
search_df = pd.DataFrame([{'rank': rank, 'filename': row['filename'], 'content_kind': row['content_kind'], 'rrf_score': row['@search.score'], 'semantic_score': row['@search.reranker_score'], 'caption': '\n'.join(caption.text for caption in row.get('@search.captions') or []), 'content': row['content']} for rank, row in enumerate(hybrid_results, start=1)])
ShowPreview(search_df)
print('Claim:', claim_id, '| Results:', len(hybrid_results), '| Vector candidates requested: 50 | Semantic: claims-semantic')

## Cleanup
Close loading/storage clients and remove only temporary ADLS staging. Durable records and original files are untouched. The question and query cells above can still run independently.

In [ ]:
for name in ['search_client', 'credential']:
    client = globals().get(name)
    if client is not None:
        client.close()
for client in globals().get('storage_clients', []):
    client.close()
for staging in globals().get('staging_folders', []):
    staging.cleanup()
print('Clients closed; temporary staging removed. Durable data was not deleted.')